utils file used to :
1- loading data 
2- definning functions 

In [0]:


# Databricks notebook source
# =============================================================================
# monitoring_utils_v3_final.py
# =============================================================================
# Shared helpers for the file-monitoring pipeline.
#
# Tasks:
#   A. Constants & mask registry
#   B. Delta Table helpers  (config + alarm log)
#   C. Date / window helpers
#   D. Filename helpers
#   E. File-system scanner
#   F. ADF pipeline client
#   G. Alarm-level logic
#
# Migration log — dict → Spark DataFrame
# ---------------------------------------
# load_usecase_config  now returns a Spark DataFrame (one row per file).
# New helpers added so every downstream step stays pure-Spark:
#
#   get_scalar_config(config_df)           → dict of usecase-level scalars
#   get_expected_files_df(config_df, t)    → DataFrame + resolved_filename col
#   list_raw_files_df(config_df)           → DataFrame of found filenames
#   write_alarm_records_df(missing_df,..)  → single-action batch Delta write
#==============================================
# Bug fixes applied in this version
# -----------------------------------
# [B1] load_usecase_config: `if not returned_df` always True for a DataFrame
#      → replaced with .count() == 0 check
# [D1] resolve_filename: `template` variable undefined when neither the
#      from/to range branch nor the MonthName branch executes
#      → initialise `template = file_template` at function start
# [E1] get_expected_files: checked for column "data_files" (never exists)
#      → now checks "file_template", "raw_data_path", "subdirectory"
# [E2] get_expected_files: `if len(missing) > 1` — off-by-one; a single
#      missing column was silently ignored
#      → corrected to `> 0`
# [E3] get_expected_files: loop accessed row["data_files"] (NameError) and
#      returned the raw template instead of the resolved filename
#      → loop now uses row["file_template"] and returns resolved filenames
# [E4] list_raw_files: signature still (base_path:str, data_files:dict)
#      → replaced with list_raw_files_df(config_df) that returns a DataFrame
# [O1] Orchestrator steps 5/7/8 used undefined `config` dict (NameError)
#      → use meta = get_scalar_config(config_df) for scalar access
# [O2] Orchestrator step 6: `found_files` never defined
#      → replaced with DataFrame anti-join
# [O3] Orchestrator step 8: per-file write loop replaced by
#      write_alarm_records_df (single Spark action)


In [0]:
%pip install azure-mgmt-datafactory azure-identity  tomlrt

In [ ]:
%sql
-- Corresponding DDL
-- truncate  table  monitoring.usecase_config;

CREATE TABLE IF NOT EXISTS monitoring.usecase_config (
usecase_name STRING NOT NULL,
frequency STRING NOT NULL, -- D / W / M
lag_days INT, -- delivery lag in days (D only)
deadline_rule INT, -- NULL=daily | weekday 0-6 | day-of-month 1-31
date_mask STRING NOT NULL, -- yyyyMMdd | yyyyMM | yyyy | MMdd
length_of_date INT,
file_frequency STRING,
file_ownership BOOLEAN,
sender_type STRING,
sftp_area_path STRING,
raw_data_path STRING NOT NULL,
pipeline_name STRING,
subdirectory STRING, -- "" for flat layout
file_template STRING NOT NULL, -- single pattern per row
updated_at TIMESTAMP DEFAULT current_timestamp()
) USING DELTA
TBLPROPERTIES (
'delta.feature.allowColumnDefaults' = 'supported',
'delta.enableChangeDataFeed' = 'true'
);


In [ ]:
%sql

-- commit 3: offset of file (lag_days)  : delay between pipeline scheduling and date of filename (not ingestion date !) involved in ingestion for specific instance/execution  of the pipeline


-- Example INSERTs :
INSERT INTO monitoring.usecase_config VALUES
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_1_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_2_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_3_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_1_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_2_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_1','D',2,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_1/','usecase_1','folder_usecase_1','file_3_usecase_1_yyyyMMdd.csv',current_timestamp()),
('usecase_2','D',1,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_2/','usecase_2','folder_usecase_2','file_1_usecase_2_from[yyyyMMdd]_to[yyyyMMdd].csv.gz',current_timestamp()),
('usecase_2','D',1,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_2/','usecase_2','folder_usecase_2','file_2_usecase_2_from[yyyyMMdd]_to[yyyyMMdd].csv.gz',current_timestamp()),
('usecase_2','D',1,NULL,'yyyyMMdd',8,'D',TRUE,'External',NULL,'/roots/usecase_2/','usecase_2','folder_usecase_2','file_3_usecase_2_from[yyyyMMdd]_to[yyyyMMdd].csv.gz',current_timestamp())


In [0]:
import re
import logging
from datetime import datetime, timedelta, date
import calendar
from typing import Union
from azure.identity import DefaultAzureCredential 
# from azure.mgmt.resource import ResourceManagementClient
# from azure.mgmt.datafactory import DataFactoryManagementClient
# from azure.mgmt.datafactory.models import *
from datetime import datetime, timedelta
import time
import tomlrt


In [ ]:
with open("pyproject.toml", "rb") as f:
    doc = tomlrt.load(f)

    main_file=doc.get("paths")["main_file"]
    utils_file=doc.get("paths")["utils_file"]
    ALARM_LEVELS=doc.get("ALARM_LEVELS")
    _MASK_TO_STRFTIME=doc.get("_MASK_TO_STRFTIME")
    _SUPPORTED_FREQUENCIES=doc.get("_SUPPORTED_FREQUENCIES")["freq"]
    _CONFIG_TABLE=doc.get("tables")["_CONFIG_TABLE"]
    _ALARM_TABLE=doc.get("tables")["_ALARM_TABLE"]

In [ ]:
import dotenv
import os


subscription_id = os.getenv("subscription_id")
resource_group= os.getenv("resource_group")
factory_name= os.getenv("factory_name")

In [0]:

# ============================================================================ #
# B. DELTA TABLE HELPERS                                                        #
# ============================================================================ #

def load_usecase_config(usecase_name: str):
    """
    Load usecase configuration from the Delta Table.

    Returns a Spark **DataFrame** with one row per expected file
    (same schema as monitoring.usecase_config, filtered to usecase_name).

    Columns
    -------
    usecase_name, frequency, lag_days, deadline_rule, date_mask,
    length_of_date, file_frequency, file_ownership, sender_type,
    sftp_area_path, raw_data_path, pipeline_name,
    subdirectory, file_template, updated_at

    Use get_scalar_config(df) to extract usecase-level scalars (frequency,
    pipeline_name, …) as a plain dict for functions that need scalar values.

    Raises
    ------
    ValueError  : usecase not found or all rows have empty file_template.
    RuntimeError: unexpected error reading from Delta Table.
    """
    if not isinstance(usecase_name, str) or not usecase_name.strip():
        raise ValueError("usecase_name must be a non-empty string.")

    usecase_name = usecase_name.strip().lower().replace(" ", "_")

    try:
        config_df = (
            spark.table(_CONFIG_TABLE)                           # noqa: F821
            .filter(f"usecase_name = '{usecase_name}'")
            .filter("file_template IS NOT NULL AND trim(file_template) != ''")
        )
        # FIX [B1]: DataFrame is always truthy — must call .count() to detect empty
        row_count = config_df.count()
    except Exception as exc:
        raise RuntimeError(
            f"Failed to query '{_CONFIG_TABLE}' for usecase '{usecase_name}': {exc}"
        ) from exc

    if row_count == 0:
        raise ValueError(
            f"Usecase '{usecase_name}' not found in '{_CONFIG_TABLE}' "
            "(or all rows have an empty file_template). "
            "Add at least one row with a non-empty file_template before running."
        )

    log.info(
        "[load_usecase_config] Loaded %d row(s) for usecase '%s'.",
        row_count, usecase_name,
    )
    return config_df


In [0]:


def get_scalar_config(config_df) -> dict:
    """
    Extract usecase-level scalar values from the config DataFrame.

    All rows share the same usecase-level fields — this function takes the
    first row and returns those columns as a plain dict, so functions that
    need scalar inputs (get_inspection_window, compute_alarm_level, …) can
    keep their existing signatures.

    Per-file columns (raw_data_path, subdirectory, file_template) are
    deliberately excluded — they belong to the row level, not the usecase level.

    Returns
    -------
    dict with keys: usecase_name, frequency, lag_days, deadline_rule,
                    date_mask, length_of_date, file_frequency, file_ownership,
                    sender_type, sftp_area_path, pipeline_name, updated_at

    Raises
    ------
    ValueError : config_df is empty.
    """
    first = config_df.select(*_SCALAR_COLS).first()
    if first is None:
        raise ValueError("config_df is empty — cannot extract scalar config.")
    # Explicitly keep only the declared scalar columns so per-file fields
    # (raw_data_path, subdirectory, file_template) cannot leak into the dict
    # regardless of what Spark's .select() returns in test mocks.
    row_dict = first.asDict()
    return {k: row_dict[k] for k in _SCALAR_COLS if k in row_dict}



In [ ]:

def write_alarm_records_df(
    missing_df,
    check_date:       datetime,
    alarm_level:      int,
    alarm_label:      str,
    usecase_name:     str,
    input_source:     str,
    usecase_deadline,
) -> int:
    """
    Write all missing-file alarms to the Delta alarm log in ONE Spark action.

    Replaces the old per-file for-loop of write_alarm_record calls.
    missing_df must contain columns: resolved_filename, raw_data_path,
    subdirectory (produced by get_expected_files_df after an anti-join).

    Returns
    -------
    int — number of alarm rows written.

    Raises
    ------
    ValueError  : unrecognised alarm_level.
    RuntimeError: Delta write failed.
    """
    if alarm_level not in ALARM_LEVELS:
        raise ValueError(
            f"Invalid alarm_level '{alarm_level}'. "
            f"Must be one of: {sorted(ALARM_LEVELS)}."
        )

    from pyspark.sql import functions as F  # noqa: F821

    alarm_df = (
        missing_df
        .withColumn("check_date",        F.lit(str(check_date)))
        .withColumn("usecase",           F.lit(usecase_name))
        .withColumn("alarm_level",       F.lit(alarm_level))
        .withColumn("alarm_label",       F.lit(alarm_label))
        .withColumn("missing_file",      F.col("resolved_filename"))
        .withColumn("expected_filename", F.col("resolved_filename"))
        .withColumn(
            "input_path",
            F.concat(
                F.lit("/mnt/"),
                F.regexp_replace(F.col("raw_data_path"), r"^/|/$", ""),
                F.when(F.col("subdirectory") != "", F.concat(F.lit("/"), F.col("subdirectory")))
                 .otherwise(F.lit("")),
            ),
        )
        .withColumn("input_source",     F.lit(input_source))
        .withColumn("usecase_deadline", F.lit(str(usecase_deadline)))
        .select(
            "check_date", "usecase", "alarm_level", "alarm_label",
            "missing_file", "input_path", "input_source",
            "expected_filename", "usecase_deadline",
        )
    )

    n = alarm_df.count()

    try:
        alarm_df.write.format("delta").mode("append").saveAsTable(_ALARM_TABLE)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to write {n} alarm record(s) for '{usecase_name}' "
            f"to '{_ALARM_TABLE}': {exc}"
        ) from exc

    log.info(
        "[write_alarm_records_df] Wrote %d alarm record(s) for '%s' (level=%d).",
        n, usecase_name, alarm_level,
    )
    return n

In [0]:

# ============================================================================ #
# C. DATE / WINDOW HELPERS                                                      #
# ============================================================================ #

def get_inspection_window(frequency: str, lag_days, sysdate: datetime):
    """
    Return (window_start, window_end) for when we expect a file to arrive.

    D lag=n → [today − n,    today − n + 1 day)
    W any   → [last Monday,  next Monday)
    M any   → [1st prev month, 1st this month)

    Both ends are timezone-aware (same tz as sysdate).
    """
    today = sysdate.date()

    if frequency == "D":
        if lag_days is None or int(lag_days) < 0:
            raise ValueError(
                f"lag_days must be a non-negative integer for daily frequency, "
                f"got {lag_days!r}."
            )
        lag   = int(lag_days)
        start = today - timedelta(days=lag)
        end   = start + timedelta(days=1)

    elif frequency == "W":
        monday = today - timedelta(days=today.weekday())
        start  = monday
        end    = monday + timedelta(days=7)

    else:  # M
        first_this_month = today.replace(day=1)
        if first_this_month.month == 1:
            start = first_this_month.replace(year=first_this_month.year - 1, month=12)
        else:
            start = first_this_month.replace(month=first_this_month.month - 1)
        end = first_this_month

    tz = sysdate.tzinfo or pytz.utc
    return (
        datetime(start.year, start.month, start.day, tzinfo=tz),
        datetime(end.year,   end.month,   end.day,   tzinfo=tz),
    )


In [0]:

def compute_deadline(
    sysdate:       datetime,
    frequency:     str,
    deadline_rule: Union[int, str, None],
    lag_days,
) -> date:
    """
    Derive the deadline date from usecase config.

    None  → daily:   deadline = (today − lag_days) + 2 days
    0–6   → weekly:  target weekday (0=Mon…6=Sun)
    1–31  → monthly: day-of-month deadline
    """
    today = sysdate.date()

    if frequency == "D":
        if lag_days is None or int(lag_days) < 0:
            raise ValueError(
                f"lag_days must be a non-negative integer for daily, got {lag_days!r}."
            )
        return today - timedelta(days=int(lag_days)) + timedelta(days=2)

    elif frequency == "W":
        if deadline_rule is None:
            raise ValueError(
                "deadline_rule is required for weekly usecases (0=Mon…6=Sun)."
            )
        target_wd = int(deadline_rule)
        if not 0 <= target_wd <= 6:
            raise ValueError(f"Weekly deadline_rule must be 0–6, got {target_wd}.")
        delta = (target_wd - today.weekday()) % 7
        return today + timedelta(days=delta or 7)

    else:  # M
        if deadline_rule is None:
            raise ValueError(
                "deadline_rule is required for monthly usecases (1–31)."
            )
        target_day = int(deadline_rule)
        if not 1 <= target_day <= 31:
            raise ValueError(f"Monthly deadline_rule must be 1–31, got {target_day}.")
        last_day = calendar.monthrange(today.year, today.month)[1]
        clamped  = min(target_day, last_day)
        if today.day <= clamped:
            return today.replace(day=clamped)
        if today.month == 12:
            ny, nm = today.year + 1, 1
        else:
            ny, nm = today.year, today.month + 1
        return date(ny, nm, min(target_day, calendar.monthrange(ny, nm)[1]))



In [0]:
# from datetime import datetime, timedelta, date

# def get_expected_file(file_formal, datamask, frequency, target_weekday, lag_days, deadline=None, previous_month=True):
#     """
#     Returns the expected reference date for a file.

#     Parameters:
#         file_formal: kept from your original signature
#         datamask: optional strftime mask for formatting the result
#         frequency: 'D', 'W', or 'M'
#         target_weekday: int from 0 (Monday) to 6 (Sunday)
#         lag_days: delay in days for daily files
#         deadline: optional date/datetime used by your process
#         previous_month: for monthly files, choose previous month if True

#     Returns:
#         date object or formatted string if datamask is provided
#     """
#     today = datetime.now().date()

#     if frequency == "D":
#         expected_date = today - timedelta(days=lag_days)

#     elif frequency == "W":
#         if not 0 <= target_weekday <= 6:
#             raise ValueError("target_weekday must be between 0 (Monday) and 6 (Sunday)")

#         # Most recent occurrence of the target weekday
#         delta_days = (today.weekday() - target_weekday) % 7
#         expected_date = today - timedelta(days=delta_days)

#     elif frequency == "M":
#         year = today.year
#         month = today.month

#         if previous_month:
#             if month == 1:
#                 year -= 1
#                 month = 12
#             else:
#                 month -= 1

#         # Use first day of the selected month as the reference date
#         expected_date = date(year, month, 1)

#     else:
#         raise ValueError("Invalid frequency. Use 'D', 'W', or 'M'.")

#     if datamask:
#         return expected_date.strftime(datamask)

#     return expected_date

In [0]:

# ============================================================================ #
# D. FILENAME HELPERS                                                           #
# ============================================================================ #

def _format_date(ref_date: Union[date, datetime], datamask: str) -> str:
    """Convert a date/datetime to string using the custom mask registry."""
    strftime_mask = _MASK_TO_STRFTIME.get(datamask)
    if not strftime_mask:
        raise ValueError(
            f"Unknown datamask '{datamask}'. "
            f"Supported: {list(_MASK_TO_STRFTIME)}."
        )
    return ref_date.strftime(strftime_mask)


def resolve_filename(
    file_template: str,
    ref_date:      Union[date, datetime],
    datamask:      str,
) -> str:
    """
    Replace datamask placeholder(s) in a filename template.

    Supported styles:
      yyyyMMdd / [yyyyMMdd]              bare and bracketed tokens
      from[yyyyMMdd]_to[yyyyMMdd]        daily range — same date both ends
      [MonthName]                        abbreviated month name (Apr, May, …)

    Examples
    --------
    >>> resolve_filename("sales_yyyyMMdd.csv", date(2025, 4, 14), "yyyyMMdd")
    'sales_20250414.csv'
    >>> resolve_filename("ESTRAZIONE_from[yyyyMMdd]_to[yyyyMMdd].csv.gz",
    ...                  date(2025, 4, 14), "yyyyMMdd")
    'ESTRAZIONE_from20250414_to20250414.csv.gz'
    """
    if not file_template or not file_template.strip():
        raise ValueError("file_template must be a non-empty string.")

    if isinstance(ref_date, datetime):
        ref_date = ref_date.date()

    # FIX [D1]: initialise template here so every branch below can reference it
    # safely, regardless of which placeholder style the template uses.
    template = file_template

    # ── Daily range mask: from[yyyyMMdd]_to[yyyyMMdd] ─────────────────────────
    if "from[yyyyMMdd]" in template and "to[yyyyMMdd]" in template:
        return template.replace("[yyyyMMdd]", ref_date.strftime("%Y%m%d"))

    # ── Month-name placeholder ─────────────────────────────────────────────────
    if "[MonthName]" in template:
        template = template.replace("[MonthName]", ref_date.strftime("%b"))

    # ── Bracketed date placeholders [yyyyMMdd] / [yyyyMM] / … ─────────────────
    for mask_token, strftime_fmt in _MASK_TO_STRFTIME.items():
        bracketed = f"[{mask_token}]"
        if bracketed in template:
            template = template.replace(bracketed, ref_date.strftime(strftime_fmt))

    # ── Bare date placeholders — longest first to avoid partial substitution ───
    for mask_token in sorted(_MASK_TO_STRFTIME, key=len, reverse=True):
        if mask_token in template:
            template = template.replace(
                mask_token, ref_date.strftime(_MASK_TO_STRFTIME[mask_token])
            )

    return template



In [0]:

def _validate_frequency(frequency: str) -> None:
    """Raise ValueError if frequency is not D / W / M."""
    if frequency not in _SUPPORTED_FREQUENCIES:
        raise ValueError(
            f"Unsupported frequency '{frequency}'. "
            f"Use one of: {_SUPPORTED_FREQUENCIES}."
        )


def get_reference_date(
    frequency:     str,
    lag_days,
    deadline_rule,
    sysdate:       datetime,
) -> date:
    """
    Compute the reference date for expected-filename generation.

    D → today − lag_days
    W → most recent occurrence of the deadline weekday (≤ today)
    M → 1st day of the previous month
    """
    today = sysdate.date()

    if frequency == "D":
        if lag_days is None or int(lag_days) < 0:
            raise ValueError(
                f"lag_days must be a non-negative integer for daily, got {lag_days!r}."
            )
        return today - timedelta(days=int(lag_days))

    elif frequency == "W":
        if deadline_rule is None:
            raise ValueError(
                "deadline_rule (weekday 0–6) is required for weekly usecases."
            )
        dr = int(deadline_rule)
        if not 0 <= dr <= 6:
            raise ValueError(f"Weekly deadline_rule must be 0–6, got {dr}.")
        return today - timedelta(days=(today.weekday() - dr) % 7)

    else:  # M
        first = today.replace(day=1)
        return (
            first.replace(year=first.year - 1, month=12)
            if first.month == 1
            else first.replace(month=first.month - 1)
        )


# ============================================================================ #
# E. FILE-SYSTEM SCANNER                                                        #
# ============================================================================ #

def get_expected_files_df(config_df, sysdate: datetime):
    """
    Add a ``resolved_filename`` column to config_df via a Spark UDF.

    The reference date is computed once on the driver from the first row's
    scalar config; the UDF then applies resolve_filename to every file_template
    row inside Spark's execution engine.

    Parameters
    ----------
    config_df : DataFrame from load_usecase_config()
    sysdate   : datetime (timezone-aware preferred)

    Returns
    -------
    DataFrame — all original columns + resolved_filename (StringType).
    The result is ready to anti-join against list_raw_files_df() output.

    Raises
    ------
    ValueError : required columns missing, or reference-date computation fails.
    """
    from pyspark.sql import functions as F   # noqa: F821
    from pyspark.sql.types import StringType # noqa: F821

    # FIX [E1]: validate correct column names
    required = {"frequency", "lag_days", "deadline_rule",
                "date_mask", "file_template", "raw_data_path", "subdirectory"}
    missing  = required - set(config_df.columns)
    if missing:
        raise ValueError(
            f"config_df is missing required columns: {sorted(missing)}."
        )

    meta = get_scalar_config(config_df)

    try:
        ref_date = get_reference_date(
            frequency     = meta["frequency"],
            lag_days      = meta["lag_days"],
            deadline_rule = meta["deadline_rule"],
            sysdate       = sysdate,
        )
    except ValueError as exc:
        raise ValueError(
            f"Cannot compute reference date for "
            f"'{meta.get('usecase_name', '?')}': {exc}"
        ) from exc

    # Pass as a literal so the UDF is stateless and serialisable across all
    # Spark / Databricks versions.
    ref_date_str = ref_date.isoformat()  # "YYYY-MM-DD"

    @F.udf(returnType=StringType())
    def _resolve_udf(file_template, date_iso, datamask):
        """
        Inlined resolve_filename so the UDF has no external dependencies —
        safe for serialisation on all Databricks runtimes.
        """
        if not file_template or not file_template.strip():
            return None
        from datetime import date as _date
        ref = _date.fromisoformat(date_iso)

        mask_map = {
            "yyyyMMdd": "%Y%m%d",
            "yyyyMM":   "%Y%m",
            "yyyy":     "%Y",
            "MMdd":     "%m%d",
        }
        # FIX [D1] replicated here: template initialised before any branch
        t = file_template

        if "from[yyyyMMdd]" in t and "to[yyyyMMdd]" in t:
            return t.replace("[yyyyMMdd]", ref.strftime("%Y%m%d"))

        if "[MonthName]" in t:
            t = t.replace("[MonthName]", ref.strftime("%b"))

        for tok, fmt in mask_map.items():
            if f"[{tok}]" in t:
                t = t.replace(f"[{tok}]", ref.strftime(fmt))

        for tok in sorted(mask_map, key=len, reverse=True):
            if tok in t:
                t = t.replace(tok, ref.strftime(mask_map[tok]))

        return t

    return config_df.withColumn(
        "resolved_filename",
        _resolve_udf(
            F.col("file_template"),
            F.lit(ref_date_str),
            F.col("date_mask"),
        ),
    )



In [0]:


def list_raw_files_df(config_df):
    """
    Scan the lake for every distinct (raw_data_path, subdirectory) pair in
    config_df and return a DataFrame of found filenames.

    dbutils.fs.ls() runs on the driver (Databricks requirement); results are
    parallelised into a Spark DataFrame so missing-file detection can be a
    native anti-join rather than a Python set look-up.

    Parameters
    ----------
    config_df : DataFrame from load_usecase_config()

    Returns
    -------
    DataFrame with columns:
        raw_data_path  (STRING)
        subdirectory   (STRING)
        found_filename (STRING)

    Raises
    ------
    ValueError : required columns missing in config_df.
    """
    from pyspark.sql.types import StructType, StructField, StringType  # noqa: F821

    required = {"raw_data_path", "subdirectory"}
    missing  = required - set(config_df.columns)
    if missing:
        raise ValueError(
            f"config_df is missing columns: {sorted(missing)}."
        )

    # Collect distinct path pairs — O(unique paths), not O(files)
    pairs = (
        config_df
        .select("raw_data_path", "subdirectory")
        .distinct()
        .collect()
    )

    found_rows = []
    for row in pairs:
        raw_path = (row["raw_data_path"] or "").strip()
        subdir   = (row["subdirectory"]  or "").strip()
        dir_path = "/mnt/" + raw_path.strip("/")
        if subdir:
            dir_path = dir_path.rstrip("/") + "/" + subdir

        try:
            entries = dbutils.fs.ls(dir_path)  # noqa: F821
            for fi in entries:
                found_rows.append((raw_path, subdir, fi.name))
        except Exception as exc:
            log.warning(
                "[list_raw_files_df] Cannot list '%s': %s", dir_path, exc
            )

    schema = StructType([
        StructField("raw_data_path",  StringType(), True),
        StructField("subdirectory",   StringType(), True),
        StructField("found_filename", StringType(), True),
    ])
    return spark.createDataFrame(found_rows, schema)  # noqa: F821


In [0]:

# ============================================================================ #
# F. ADF PIPELINE CLIENT #
# ============================================================================ #

def retrieve_last_adf_run(
    pipeline_name: str,
    freq: str,
    sysdate,
    look_back_days: int = 1
    
):
    """
    Query Azure Data Factory for the most recent run of *pipeline_name*.

    Parameters
    ----------
    pipeline_name : ADF pipeline name (must match exactly).
    freq : usecase frequency (D / W / M); controls look-back window.
    look_back_days : fallback look-back only when freq is not in _ADF_LOOKBACK_DAYS.

    Returns
    -------
    PipelineRun object (has .run_start, .status, .run_id, .pipeline_name)
    or None if no run is found.

    Raises
    ------
    ValueError : pipeline_name is empty or freq is unsupported.
    RuntimeError : Azure SDK import failed or ADF query raised an exception.
    """
    if not pipeline_name or not pipeline_name.strip():
        raise ValueError("pipeline_name must be a non-empty string.")
    _validate_frequency(freq)

    try:
        from azure.identity import DefaultAzureCredential  # noqa: F401
        from azure.mgmt.datafactory import DataFactoryManagementClient
    except ImportError as exc:
        raise RuntimeError(
            "Azure SDK not installed. "
            "Run: pip install azure-identity azure-mgmt-datafactory"
        ) from exc

    lookback = _ADF_LOOKBACK_DAYS.get(freq, look_back_days)
    end_time = sysdate 
    start_time = sysdate - timedelta(days=lookback)
    # sysdate=sysdate.d

    try:
        from azure.identity import DefaultAzureCredential
        credential = DefaultAzureCredential(
            exclude_environment_credential=False,
            exclude_workload_identity_credential=True,
            exclude_shared_token_cache_credential=True,
            exclude_visual_studio_code_credential=True,
            exclude_cli_credential=True,
            exclude_powershell_credential=True,
            exclude_interactive_browser_credential=True,
        )
        client = DataFactoryManagementClient(credential, _SUBSCRIPTION_ID)
        response = client.pipeline_runs.query_by_factory(
            _RESOURCE_GROUP,
            _FACTORY_NAME,
            filter_parameters={
                "filters": [
                    {"operand": "PipelineName", "operator": "In", "values": [pipeline_name]},
                    {"operand": "LatestOnly", "operator": "Equals", "values": ["true"]},
                ],
                "orderBy": [{"orderBy": "RunStart", "order": "DESC"}],
                "last_updated_after": start_time,
                "last_updated_before": end_time,
            },
        )
    except Exception as exc:
        raise RuntimeError(
            f"ADF query failed for pipeline '{pipeline_name}': {exc}"
        ) from exc

    runs = response.value
    if not runs:
        log.info(
            "[retrieve_last_adf_run] No runs found for '%s' in last %d day(s).",
            pipeline_name, lookback,
        )
        return None

    return max(runs, key=lambda r: r.run_start)

 

In [0]:

# ============================================================================ #
# G. ALARM-LEVEL LOGIC #
# ============================================================================ #

def compute_alarm_level(
    sysdate: datetime,
    last_run,
    frequency: str,
    deadline_rule: Union[int, str, None],
    lag_days,
) -> int:
    """
    Determine the alarm level when one or more files are missing.

    Classification:
    1 → sysdate < next expected run (within window, informational)
    2 → next_run ≤ sysdate ≤ deadline (stalled, warning)
    3 → sysdate > deadline (past deadline — critical)

    Raises
    ------
    ValueError : unsupported frequency.
    """
    _validate_frequency(frequency)

    tz = sysdate.tzinfo or pytz.utc

    # ── Next expected ADF run ──────────────────────────────────────────────────
    # getattr used to get start_date from object
    run_start = getattr(last_run, "run_start", None) if last_run else None
    print(f"starting run ${run_start} ================")
    if run_start:
        if run_start.tzinfo is None:
            run_start = pytz.utc.localize(run_start)
        freq_days = {"D": 1, "W": 7, "M": 30}
        next_run = run_start + timedelta(days=freq_days.get(frequency,"D"))
        print(f"next_run : {next_run} ")

    else:
        log.info("[compute_alarm_level] No prior ADF run found — defaulting to level ≥ 2.")
        next_run = sysdate  # forces level ≥ 2 immediately

    # ── Deadline ───────────────────────────────────────────────────────────────
    try:
        deadline_date = compute_deadline(sysdate, frequency, deadline_rule, lag_days)
        print(f"deadline {deadline_date}")
    except Exception as exc:
        log.warning(
            "[compute_alarm_level] Could not compute deadline (%s). Defaulting to level 2.", exc
        )
        return 2

    deadline_dt = datetime(
        deadline_date.year, deadline_date.month, deadline_date.day,
        23, 59, 59, tzinfo=tz,
    )
    sysdate=datetime.now(pytz.utc)
    # ── Classify ───────────────────────────────────────────────────────────────
    if sysdate.date() > deadline_date:
        print(f"sysdate.date typs is {sysdate.date()}\n deadline date type {deadline_date}")
        return 3
    if sysdate < next_run:
        print(f"sysdate is {type(sysdate)} next_run is {type(next_run)}")
        return 1
    return 2

